# ChArUco Board Creation and Detection Test

Generate a printable ChArUco calibration board image and verify that OpenCV
can detect it from a sample camera frame.

**Steps:**
1. **Create board** — Renders a high-DPI PNG of a 10×7 ChArUco board
   (DICT_4X4_50) at the physical dimensions used in the Rat Lockbox rig.
   The output PNG (`examples/charuco_board.png`) can be printed and used as a
   calibration target.
2. **Test detection (modern API)** — Uses `cv2.aruco.CharucoDetector` to
   detect corners in a test frame image.
3. **Test detection (legacy API)** — Alternative detection flow using the
   older `detectMarkers` + `interpolateCornersCharuco` pipeline for
   compatibility with older OpenCV builds.

In [13]:
import cv2
import cv2.aruco as aruco

# ===== Board parameters (EDIT THESE) =====
squaresX = 10            # number of chessboard squares in X
squaresY = 7            # number of chessboard squares in Y
squareLength = 0.010     # meters
markerLength = 0.005     # meters
dpi = 600               # print DPI

dictionary = aruco.getPredefinedDictionary(aruco.DICT_4X4_50)

# ===== Create board =====
board = aruco.CharucoBoard(
    (squaresX, squaresY),
    squareLength,
    markerLength,
    dictionary
)

# ===== Compute image size in pixels =====
width_m  = squaresX * squareLength
height_m = squaresY * squareLength

px_per_meter = dpi / 0.0254
width_px  = int(width_m * px_per_meter)
height_px = int(height_m * px_per_meter)

# ===== Generate image =====
img = board.generateImage((width_px, height_px), marginSize=0, borderBits=1)

cv2.imwrite("charuco_board.png", img)
print("Saved charuco_board.png")

Saved charuco_board.png


## Test detection on frame

In [2]:
import cv2
import numpy as np

# Simple version for quick testing
image_path = "test_frame.png"
# Load image
img = cv2.imread(image_path)
if img is None:
    print(f"Error: Could not load {image_path}")

# Try different dictionary types
dict_types = ['DICT_4X4_50']

for dict_name in dict_types:
    print(f"\nTrying {dict_name}...")
    
    # Get dictionary
    aruco_dict = cv2.aruco.getPredefinedDictionary(getattr(cv2.aruco, dict_name))
    
    # Create board (adjust dimensions as needed)
    board = cv2.aruco.CharucoBoard((10, 7), 0.01, 0.005, aruco_dict)
    
    # Detect
    detector = cv2.aruco.CharucoDetector(board)
    corners, ids, marker_corners, marker_ids = detector.detectBoard(img)
    
    if corners is not None and len(corners) > 0:
        print(f"✓ Success! Found {len(corners)} corners")
        
        # Draw results
        result = img.copy()
        result = cv2.aruco.drawDetectedCornersCharuco(result, corners, ids)
        
        # Show image
        cv2.imshow('Charuco Detection', result)
        cv2.waitKey(0)
        cv2.destroyAllWindows()
    else:
        print(f"✗ No corners detected")


Trying DICT_4X4_50...
✓ Success! Found 32 corners


## Test detection on frame (v2)

In [3]:
import cv2
import cv2.aruco as aruco

img = cv2.imread("test_frame.png")
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

dictionary = aruco.getPredefinedDictionary(aruco.DICT_4X4_50)

corners, ids, _ = aruco.detectMarkers(gray, dictionary)

print("Markers detected:", 0 if ids is None else len(ids))

board = aruco.CharucoBoard(
    (10, 7),          # squaresX, squaresY
    0.010,            # squareLength (meters)
    0.005,            # markerLength (meters)
    dictionary
)
corners = [c.astype("float32") for c in corners]
ids = ids.astype("int32")

if ids is not None:
    ret, charuco_corners, charuco_ids = aruco.interpolateCornersCharuco(
        corners, ids, gray, board
    )
    print("Charuco corners:", ret)
else:
    print("No markers")


Markers detected: 25
Charuco corners: 32
